# Housing Price Prediction - Exploratory Data Analysis (EDA)
==========================================================

This notebook explores the Housing dataset to understand the features, their distributions, and their relationships with the target variable (`price`).

## Objectives:
1. Load and inspect the raw dataset.
2. Check for missing values, duplicates, and data type correctness.
3. Analyze the distribution of the target variable (`price`).
4. Explore continuous features and their relationships with housing prices.
5. Examine categorical features and how they impact prices.
6. Analyze correlations among numeric features.

## 1. Setup and Loading Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Load the dataset
data_path = os.path.join("..", "data", "raw", "Housing.csv")
df = pd.read_csv(data_path)
print(f"Dataset loaded successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

## 2. Basic Dataset Inspection

In [ ]:
# View first few rows
df.head()

In [ ]:
# Check data types and non-null values
df.info()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

In [ ]:
# Summary statistics for numerical variables
df.describe()

## 3. Distribution of the Target Variable (`price`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram & KDE
sns.histplot(df["price"], kde=True, color="#4C72B0", ax=axes[0])
axes[0].set_title("Distribution of House Prices")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Frequency")

# Box plot to identify outliers
sns.boxplot(x=df["price"], color="#55A868", ax=axes[1])
axes[1].set_title("Boxplot of House Prices")
axes[1].set_xlabel("Price ($)")

plt.tight_layout()
plt.show()

**Observations on Price:**
- The distribution of housing prices is right-skewed (positively skewed).
- There are visible outliers on the higher end of the price spectrum, with several properties priced above $10 million.

## 4. Continuous Features vs. Price

In [ ]:
# Scatter plot for Area vs Price with trendline
sns.regplot(data=df, x="area", y="price", scatter_kws={"alpha":0.5, "color":"#4C72B0"}, line_kws={"color":"red"})
plt.title("Area vs. Price")
plt.xlabel("Area (sq ft)")
plt.ylabel("Price ($)")
plt.show()

In [ ]:
# Bedrooms, Bathrooms, and Stories vs Price using boxplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(data=df, x="bedrooms", y="price", palette="Blues", ax=axes[0])
axes[0].set_title("Bedrooms vs. Price")
axes[0].set_xlabel("Number of Bedrooms")
axes[0].set_ylabel("Price ($)")

sns.boxplot(data=df, x="bathrooms", y="price", palette="Greens", ax=axes[1])
axes[1].set_title("Bathrooms vs. Price")
axes[1].set_xlabel("Number of Bathrooms")
axes[1].set_ylabel("Price ($)")

sns.boxplot(data=df, x="stories", y="price", palette="Purples", ax=axes[2])
axes[2].set_title("Stories vs. Price")
axes[2].set_xlabel("Number of Stories")
axes[2].set_ylabel("Price ($)")

plt.tight_layout()
plt.show()

**Observations on Continuous/Numerical Features:**
- **Area:** There is a clear positive linear relationship between house area and price. As square footage increases, the price generally increases.
- **Bedrooms:** Houses with 3, 4, or 5 bedrooms command higher median prices compared to 1 or 2 bedrooms, though 6-bedroom houses show a slight dip (possibly due to smaller overall area or other compounding factors).
- **Bathrooms:** Prices rise significantly with the number of bathrooms.
- **Stories:** Houses with more stories tend to have higher prices.

## 5. Categorical Features vs. Price

In [ ]:
categorical_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.boxplot(data=df, x=col, y="price", palette="Set2", ax=axes[i])
    axes[i].set_title(f"{col.capitalize()} vs. Price")
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel("Price ($)")

# Hide the remaining empty subplots
for j in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Observations on Categorical Features:**
- Properties facing a **main road**, having a **guestroom**, a **basement**, **hot water heating**, **air conditioning**, or located in a **preferred area** consistently command higher median prices.
- **Air conditioning** and **preferred area** show particularly pronounced differences in median price.
- **Furnishing status** also plays a significant role: furnished homes command the highest median price, followed by semi-furnished, with unfurnished homes being the cheapest.

## 6. Correlation Analysis

In [ ]:
# Preprocess binary & categorical features temporarily to compute correlations
df_corr = df.copy()
binary_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
for col in binary_cols:
    df_corr[col] = df_corr[col].map({"yes": 1, "no": 0})

df_corr = pd.get_dummies(df_corr, columns=["furnishingstatus"], drop_first=True, dtype=int)

# Compute correlation matrix
corr_matrix = df_corr.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap")
plt.show()

**Key Correlation Takeaways:**
- The features most strongly correlated with `price` are:
  1. `area` (0.54)
  2. `bathrooms` (0.52)
  3. `airconditioning` (0.45)
  4. `stories` (0.42)
  5. `bedrooms` (0.37)
- Multicollinarity is relatively low, meaning a linear regression model should not suffer heavily from redundant predictor features.

## 7. Conclusions & Next Steps

1. **Feature Importance:** `area`, `bathrooms`, `airconditioning`, and `stories` are key factors determining housing prices in this dataset.
2. **Linear Modeling Feasibility:** Since multicollinearity is low and strong linear correlations exist between multiple features and price, a **Linear Regression** model is a solid baseline choice.
3. **Preprocessing Pipeline:** Categorical columns must be properly encoded (yes/no to 1/0, and one-hot encoding for furnishing status). Features should be scaled due to scale discrepancies (e.g. `area` in thousands vs. `parking` in single digits).